# Public release note

This is a sanitized public copy of `omniASR_20_K2_KenLM_V6_text_corpus(1).ipynb`. Cell outputs, execution counters,
Colab user metadata, widget state, embedded display payloads, private storage
paths, and internal run identifiers were removed before publication. The
experiment source is preserved, but public placeholders must be configured from
your own generated contracts before rerunning dependent stages. See the repository
README and `docs/REPRODUCIBILITY.md` for prerequisites, data access, execution
order, expected artifacts, and the English explanation of this experiment.

Never paste credentials into a notebook. Use Colab Secrets or environment
variables (`HF_TOKEN`, `KAGGLE_USERNAME`, and `KAGGLE_KEY`).


# omniASR — 20.K2 : corpus texte KenLM V6

Cette étape consomme le `PASS_READY_FOR_20_K2` immuable de **20.K1** et construit, pour `sw`, `kik`, `kln`, `luo`, `som` et `mas`, un corpus texte normalisé et dédupliqué pour les futurs KenLM V6.

## Avant de lancer

1. Ouvrir ce notebook dans un **nouvel environnement Colab avec RAM augmentée**. Le GPU n'est pas utilisé par 20.K2.
2. Dans **Secrets Colab**, créer `HF_TOKEN`, activer l'accès de ce notebook et utiliser un jeton Hugging Face en lecture seule. Ne pas téléverser de fichier de jeton.
3. Vérifier que Google Drive contient le PASS 20.K1 et les caches V4/V5 précédemment sauvegardés.
4. Exécuter la cellule **20.K2** ci-dessous. Elle monte Drive elle-même.

## Reprise

En cas de déconnexion, relancer exactement la même cellule. Les candidats et sorties validés sont repris par langue. Les fichiers candidats intermédiaires sont supprimés du Drive une fois le corpus final publié, afin d'éviter une seconde copie volumineuse.

## Résultat attendu

La dernière sortie doit afficher :

`STATUT 20.K2 : PASS_READY_FOR_20_K3_KENLM_BUILD`

Le corpus final est sauvegardé sous `dataset_caches/kenlm_v6_text_20_K2/<run_id>/` dans le projet Drive. Aucun audio n'est lu ou copié.


In [ ]:
# 20.K2 — Corpus texte KenLM V6 : extraction, normalisation et deduplication
#
# Cette cellule consomme uniquement le PASS immuable de 20.K1 obtenu par
# l'utilisateur. Elle lit les colonnes texte des caches locaux et de quelques
# sources distantes explicitement approuvees par 20.K1. Aucun audio n'est lu,
# decode ou copie. Les sorties contiennent uniquement du texte normalise et sa
# provenance minimale.

from __future__ import annotations

import csv
import gc
import gzip
import hashlib
import importlib.metadata
import importlib.util
import io
import json
import math
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import unicodedata
from collections import Counter, defaultdict
from collections.abc import Iterable, Iterator, Mapping
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


# ---------------------------------------------------------------------------
# 0. Dependances texte/metadata seulement
# ---------------------------------------------------------------------------

REQUIRED_MODULES = {
    "numpy": "numpy>=1.24.0",
    "pyarrow": "pyarrow>=14.0.0",
    "pandas": "pandas>=2.0.0",
    "huggingface_hub": "huggingface_hub>=0.30.0",
    "ftfy": "ftfy>=6.2.0",
    "regex": "regex>=2023.12.25",
    "rapidfuzz": "rapidfuzz>=3.6.0",
    "duckdb": "duckdb>=1.1.0",
}

MINIMUM_VERSIONS = {
    "numpy": (1, 24, 0),
    "pyarrow": (14, 0, 0),
    "pandas": (2, 0, 0),
    "huggingface_hub": (0, 30, 0),
    "ftfy": (6, 2, 0),
    "regex": (2023, 12, 25),
    "rapidfuzz": (3, 6, 0),
    "duckdb": (1, 1, 0),
}


def numeric_version(value: str) -> tuple[int, int, int]:
    numbers = [int(part) for part in re.findall(r"\d+", value)[:3]]
    return tuple((numbers + [0, 0, 0])[:3])

missing = []
for module_name, requirement in REQUIRED_MODULES.items():
    installed = importlib.util.find_spec(module_name) is not None
    if installed:
        try:
            installed = numeric_version(
                importlib.metadata.version(module_name)
            ) >= MINIMUM_VERSIONS[module_name]
        except importlib.metadata.PackageNotFoundError:
            installed = False
    if not installed:
        missing.append(requirement)
if missing:
    print("Installation des dependances text-only :", missing)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing],
        check=True,
    )

import duckdb
import ftfy
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import regex as uregex
from ftfy.badness import badness as ftfy_badness
from huggingface_hub import HfApi, HfFileSystem
from rapidfuzz.fuzz import ratio as edit_ratio


# ---------------------------------------------------------------------------
# 1. Drive, secrets et contrat fige de 20.K1
# ---------------------------------------------------------------------------

try:
    from google.colab import drive

    if not Path("/content/persistent_storage").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

FT_ROOT = Path(
    os.environ.get(
        "AFRIVOICES_FT_ROOT",
        "/content/afrivoices_project/"
        "finetune_runs/experiment_run",
    )
)
PROJECT_ROOT = FT_ROOT.parent.parent
K1_ROOT = FT_ROOT / "reports/kenlm_v6_20_K1"
REPORT_ROOT = FT_ROOT / "reports/kenlm_v6_20_K2"
CORPUS_ROOT = PROJECT_ROOT / "dataset_caches/kenlm_v6_text_20_K2"
CACHE_ROOT = PROJECT_ROOT / "dataset_caches"

EXPECTED_K1_RUN_ID = "REPLACE_WITH_LOCAL_RUN_ID"
EXPECTED_K1_REPORT_SHA256 = (
    "REPLACE_WITH_LOCAL_SHA256"
)

LANGUAGES = ("sw", "kik", "kln", "luo", "som", "mas")
LANGUAGE_ALIASES = {
    "sw": {"sw", "swa", "swh", "swahili", "kiswahili"},
    "kik": {"kik", "ki", "kikuyu", "gikuyu"},
    "kln": {"kln", "sgc", "niq", "kalenjin", "kipsigis", "nandi"},
    "luo": {"luo", "dholuo"},
    "som": {"som", "so", "somali", "soomali", "af-soomaali"},
    "mas": {"mas", "maasai", "maa", "saq"},
}
LANGUAGE_DIR_CODES = {
    "sw": {"sw", "swa", "swh"},
    "kik": {"kik"},
    "kln": {"kln"},
    "luo": {"luo"},
    "som": {"som"},
    "mas": {"mas", "saq"},
}

TEXT_PRIORITY = (
    "text_norm",
    "text",
    "transcription",
    "transcript",
    "actualSentence",
    "sentence",
    "raw_transcription",
)
AUDIO_COLUMN_PATTERNS = (
    "audio",
    "waveform",
    "samples",
    "sample_rate",
    "sampling_rate",
    "speech_array",
)
FORBIDDEN_SPLIT_PATTERN = re.compile(
    r"(?:^|[/_.=-])(dev|valid|validation|local_test|test|eval)(?:$|[/_.=-])",
    re.IGNORECASE,
)

NORMALIZER_VERSION = "afrivoices-kenlm-v6-nfc-ftfy-safe-20260715"
PIPELINE_REVISION = "afrivoices-kenlm-v6-20k2-r6-20260715"
NORMALIZATION_COMPATIBILITY_MIN = 0.85
NORMALIZATION_REJECTION_MAX = 0.02
NORMALIZATION_IDEMPOTENCE_MIN = 0.9999
MAX_WORDS_PER_LINE = 80
MIN_NEAR_WORDS = 12
MIN_NEAR_CHARS = 80
MINHASH_COMPONENTS = 16
MINHASH_BANDS = 8
NEAR_SHORT_EDIT = 0.95
NEAR_SHORT_JACCARD = 0.85
NEAR_LONG_EDIT = 0.93
NEAR_LONG_JACCARD = 0.80
HELDOUT_EDIT = 0.98
HELDOUT_JACCARD = 0.94
HASH_PERSON = b"K2SHING"

REMOTE_SOURCE_RULES = {
    "fleurs": {
        "repo_id": "google/fleurs",
        "languages": {
            "sw": {"config": "sw_ke", "mode": "text"},
            "luo": {"config": "luo_ke", "mode": "text"},
            "som": {"config": "so_so", "mode": "text"},
        },
        "text_fields": ["transcription", "raw_transcription"],
        "priority": 30,
        "family": "fleurs",
    },
    "kenspeech_text": {
        "repo_id": "Kencorpus/KenSpeech",
        "languages": {"sw": {"filename": "transcripts_only.csv", "mode": "text"}},
        "text_fields": ["transcript"],
        "priority": 25,
        "family": "kencorpus_kenspeech",
    },
    "kenpos_dholuo": {
        "repo_id": "Kencorpus/KenPOS",
        "languages": {"luo": {"config": "dho", "mode": "tokens"}},
        "token_fields": ["token"],
        "priority": 30,
        "family": "kencorpus_kenpos",
    },
}

CANDIDATE_SCHEMA = pa.schema(
    [
        ("record_id", pa.string()),
        ("language", pa.string()),
        ("source_id", pa.string()),
        ("source_family", pa.string()),
        ("source_revision", pa.string()),
        ("source_split", pa.string()),
        ("source_path", pa.string()),
        ("source_row_id", pa.string()),
        ("source_priority", pa.int16()),
        ("segment_index", pa.int16()),
        ("text_norm", pa.string()),
        ("exact_sha256", pa.string()),
        ("word_count", pa.int16()),
        ("char_count", pa.int32()),
        ("mojibake_status", pa.string()),
        ("contains_digit", pa.bool_()),
        ("quality_flags", pa.string()),
    ]
)


def get_colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata

        value = userdata.get(name)
        return str(value).strip() if value else None
    except Exception:
        return None


HF_TOKEN = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    or get_colab_secret("HF_TOKEN")
)
SECRET_VALUES = [str(HF_TOKEN)] if HF_TOKEN else []


def redact(value: Any) -> str:
    text = str(value)
    for secret in SECRET_VALUES:
        if secret:
            text = text.replace(secret, "<REDACTED>")
    text = re.sub(
        r"(?i)(authorization|token|key)=?[^\s,;]+",
        r"\1=<REDACTED>",
        text,
    )
    return text[:1000]


def canonical_json(value: Any) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path: Path, block_size: int = 8 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(value: Any, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, default=str) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)
    print("json ->", path)


def read_small_json(path: Path, maximum_bytes: int = 64 << 20) -> dict[str, Any]:
    path = Path(path)
    assert path.is_file(), path
    assert not path.is_symlink(), f"Lien symbolique refuse : {path}"
    assert 0 < path.stat().st_size <= maximum_bytes, path
    payload = json.loads(path.read_text(encoding="utf-8"))
    assert isinstance(payload, dict), path
    return payload


def require_child(path: Path | str, parent: Path | str) -> Path:
    path = Path(path)
    parent = Path(parent)
    assert path.exists(), path
    resolved_path = path.resolve()
    resolved_parent = parent.resolve()
    assert os.path.commonpath([str(resolved_path), str(resolved_parent)]) == str(
        resolved_parent
    ), (path, parent)
    return resolved_path


def is_text_type(data_type: pa.DataType) -> bool:
    if pa.types.is_dictionary(data_type):
        data_type = data_type.value_type
    return pa.types.is_string(data_type) or pa.types.is_large_string(data_type)


def is_audio_column(name: str) -> bool:
    lowered = name.lower()
    return any(pattern in lowered for pattern in AUDIO_COLUMN_PATTERNS)


def parquet_projection_sha256(path: Path, columns: list[str]) -> tuple[str, int]:
    """Empreinte uniquement les colonnes projetees, jamais les bytes audio."""
    assert columns and not any(is_audio_column(name) for name in columns)
    parquet_file = pq.ParquetFile(path, memory_map=False, pre_buffer=False)
    digest = hashlib.sha256()
    digest.update(canonical_json(columns).encode("utf-8"))
    rows = 0
    for batch in parquet_file.iter_batches(
        batch_size=8192,
        columns=columns,
        use_threads=False,
    ):
        assert batch.schema.names == columns
        values = batch.to_pydict()
        for index in range(batch.num_rows):
            for column in columns:
                value = values[column][index]
                if value is None:
                    digest.update(b"\x00")
                else:
                    encoded = unicodedata.normalize("NFC", str(value)).encode("utf-8")
                    digest.update(b"\x01")
                    digest.update(len(encoded).to_bytes(8, "little"))
                    digest.update(encoded)
            rows += 1
    assert rows == parquet_file.metadata.num_rows
    return digest.hexdigest(), rows


assert Path("/content/persistent_storage").exists(), "Monter Google Drive."
assert FT_ROOT.is_dir(), FT_ROOT
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
CORPUS_ROOT.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# 2. Validation stricte du PASS 20.K1
# ---------------------------------------------------------------------------


def validate_k1() -> tuple[dict[str, Any], dict[str, Any], dict[str, Any]]:
    latest_path = K1_ROOT / "LATEST_PASS.json"
    latest = read_small_json(latest_path)
    assert latest.get("cell") == "20.K1"
    assert latest.get("status") == "PASS_READY_FOR_20_K2"
    assert latest.get("run_id") == EXPECTED_K1_RUN_ID, latest.get("run_id")
    assert latest.get("report_sha256") == EXPECTED_K1_REPORT_SHA256
    assert latest.get("audio_downloaded") is False
    assert latest.get("audio_downloaded_bytes") == 0
    assert latest.get("dataset_rows_read") == 0
    assert latest.get("dataset_row_payload_downloaded_bytes") == 0

    run_dir = require_child(K1_ROOT / EXPECTED_K1_RUN_ID, K1_ROOT)
    report_path = require_child(latest["report"], run_dir)
    complete_path = require_child(latest["complete"], run_dir)
    assert report_path == (
        run_dir / "kenlm_text_source_inventory_20_K1.json"
    ).resolve()
    assert complete_path == (run_dir / "_COMPLETE.json").resolve()
    assert sha256_file(report_path) == EXPECTED_K1_REPORT_SHA256
    assert sha256_file(complete_path) == latest["complete_sha256"]

    report = read_small_json(report_path)
    complete = read_small_json(complete_path)
    for payload in (report, complete):
        assert payload.get("cell") == "20.K1"
        assert payload.get("status") == "PASS_READY_FOR_20_K2"
        assert payload.get("run_id") == EXPECTED_K1_RUN_ID
        assert payload.get("contract_sha256") == latest["contract_sha256"]
    assert complete.get("metadata_only") is True
    assert complete.get("audio_downloaded_bytes") == 0
    assert complete.get("dataset_rows_read") == 0
    assert complete.get("dataset_row_payload_downloaded_bytes") == 0
    assert sha256_bytes(
        canonical_json(report["contract"]).encode("utf-8")
    ) == report["contract_sha256"]

    expected_artifacts = {
        "kenlm_text_source_inventory_20_K1.json",
        "hf_source_inventory_20_K1.csv",
        "language_coverage_20_K1.csv",
        "kaggle_file_inventory_20_K1.csv",
    }
    assert set(complete["artifacts"]) == expected_artifacts
    for name, metadata in complete["artifacts"].items():
        artifact = require_child(run_dir / name, run_dir)
        assert artifact.stat().st_size == metadata["bytes"]
        assert sha256_file(artifact) == metadata["sha256"]

    snapshot = report["v5_cache_snapshot"]
    snapshot_contract = report["contract"]["local_gate_snapshot"][
        "v5_cache_snapshot"
    ]
    assert canonical_json(snapshot) == canonical_json(snapshot_contract)
    assert set(snapshot) == set(LANGUAGES)
    return latest, report, snapshot


K1_LATEST, K1_REPORT, K1_CACHE_SNAPSHOT = validate_k1()
print("20.K1 verifie :", EXPECTED_K1_RUN_ID, EXPECTED_K1_REPORT_SHA256[:16])


# ---------------------------------------------------------------------------
# 3. Selection des caches locaux et audit text-only de lm_text
# ---------------------------------------------------------------------------


def choose_text_column(schema: pa.Schema) -> str:
    lower_to_original: dict[str, str] = {}
    for name in schema.names:
        lowered = name.lower()
        assert lowered not in lower_to_original, (
            "Colonnes ambigues par la casse",
            schema.names,
        )
        lower_to_original[lowered] = name
    for candidate in TEXT_PRIORITY:
        original = lower_to_original.get(candidate.lower())
        if original is not None:
            assert not is_audio_column(original)
            assert is_text_type(schema.field(original).type), (
                original,
                schema.field(original).type,
            )
            return original
    raise AssertionError(f"Aucune colonne texte reconnue : {schema.names}")


def choose_optional_column(schema: pa.Schema, candidates: Iterable[str]) -> str | None:
    lower_to_original = {name.lower(): name for name in schema.names}
    for candidate in candidates:
        if candidate.lower() in lower_to_original:
            return lower_to_original[candidate.lower()]
    return None


def validate_selected_caches(
    snapshot: dict[str, Any],
) -> dict[str, dict[str, Any]]:
    selected: dict[str, dict[str, Any]] = {}
    for language in LANGUAGES:
        rows = snapshot[language]
        chosen = [row for row in rows if row.get("selected_for_20_K2") is True]
        assert len(chosen) == 1, (language, "cache selectionne", len(chosen))
        row = chosen[0]
        assert row.get("ready") is True
        assert row.get("parquet_rows_read") == 0
        cache_root = require_child(row["path"], CACHE_ROOT)
        assert cache_root.parent == CACHE_ROOT.resolve()
        assert ".partial" not in cache_root.name.lower()
        complete_path = cache_root / "_COMPLETE.json"
        assert complete_path.is_file()
        assert sha256_file(complete_path) == row["complete_sha256"]

        # Rejouer l'inventaire des petites partitions train figees en K1.
        train_parts: list[Path] = []
        train_inventory: list[dict[str, Any]] = []
        for directory_raw in row["train_directories"]:
            directory = require_child(directory_raw, cache_root)
            relative = directory.relative_to(cache_root)
            assert "split=train" in relative.parts, relative
            assert not FORBIDDEN_SPLIT_PATTERN.search(str(relative)), relative
            language_parts = [
                part for part in relative.parts if part.startswith("language=")
            ]
            assert len(language_parts) == 1, relative
            code = language_parts[0].split("=", 1)[1].lower().split("_", 1)[0]
            assert code in LANGUAGE_DIR_CODES[language], (language, code)
            for path in sorted(directory.rglob("*.parquet")):
                path = require_child(path, cache_root)
                assert not path.is_symlink()
                assert path.stat().st_size > 0
                train_parts.append(path)
                train_inventory.append(
                    {
                        "path": str(path.relative_to(cache_root)),
                        "size_bytes": path.stat().st_size,
                    }
                )
        assert len(train_parts) == row["train_parquet_parts"]
        assert sha256_bytes(
            canonical_json(train_inventory).encode("utf-8")
        ) == row["train_part_inventory_sha256"]

        # Les caches ANV possedent normalement lm_text, qui contient aussi les
        # longues transcriptions. Le cache swahili V4 n'en possede pas : dans
        # ce cas, les Parquets split=train deja figes par 20.K1 sont une source
        # textuelle equivalente. PyArrow ne projettera que texte/ID/metadata ;
        # aucune colonne audio ne sera lue.
        lm_root = cache_root / "lm_text"
        if lm_root.is_dir():
            lm_files = sorted(lm_root.rglob("*.parquet"))
            assert lm_files, (language, "lm_text present mais vide", lm_root)
            local_text_mode = "lm_text"
            local_text_root = lm_root
            local_text_files = lm_files
        else:
            assert language == "sw", (
                "lm_text obligatoire pour les caches ANV",
                language,
                lm_root,
            )
            local_text_mode = "train_parquet_projection"
            local_text_root = cache_root
            local_text_files = train_parts
        assert local_text_files, (language, "aucune source texte locale")

        local_text_plan = []
        local_text_inventory = []
        for path in local_text_files:
            path = require_child(path, local_text_root)
            assert not path.is_symlink()
            assert path.stat().st_size > 0
            relative_to_cache = path.relative_to(cache_root)
            path_proves_train = local_text_mode == "train_parquet_projection"
            if path_proves_train:
                assert "split=train" in relative_to_cache.parts, relative_to_cache
            assert not FORBIDDEN_SPLIT_PATTERN.search(str(relative_to_cache)), path
            parquet_file = pq.ParquetFile(
                path,
                memory_map=False,
                pre_buffer=False,
            )
            schema = parquet_file.schema_arrow
            assert parquet_file.metadata.num_rows > 0, path
            text_column = choose_text_column(schema)
            if path_proves_train:
                # La partition physique figée par K1 fait autorité. Ne pas
                # interpréter un éventuel source_split provenant du dataset
                # d'origine, qui peut contenir un libellé non égal à "train".
                is_dev_column = None
                split_column = None
            else:
                is_dev_column = choose_optional_column(schema, ("is_dev",))
                split_column = choose_optional_column(
                    schema,
                    ("split", "source_split", "dataset_split"),
                )
            assert path_proves_train or is_dev_column or split_column, (
                "source texte sans preuve train/dev",
                language,
                path,
                schema.names,
            )
            language_column = (
                None
                if path_proves_train
                else choose_optional_column(
                    schema,
                    ("language", "lang", "locale"),
                )
            )
            id_column = choose_optional_column(
                schema,
                ("sample_id", "source_id", "original_id", "id"),
            )
            requested = [text_column]
            for optional in (
                is_dev_column,
                split_column,
                language_column,
                id_column,
            ):
                if optional and optional not in requested:
                    requested.append(optional)
            assert not any(is_audio_column(name) for name in requested), requested
            projection_sha256, projection_rows = parquet_projection_sha256(
                path,
                requested,
            )
            record = {
                "path": str(path),
                "relative_path": str(path.relative_to(cache_root)),
                "source_mode": local_text_mode,
                "path_proves_train": path_proves_train,
                "size_bytes": path.stat().st_size,
                "text_projection_sha256": projection_sha256,
                "rows_from_footer": int(parquet_file.metadata.num_rows),
                "schema_names": list(schema.names),
                "text_column": text_column,
                "is_dev_column": is_dev_column,
                "split_column": split_column,
                "language_column": language_column,
                "id_column": id_column,
                "requested_columns": requested,
                "audio_columns_requested": [],
            }
            assert projection_rows == record["rows_from_footer"]
            local_text_plan.append(record)
            local_text_inventory.append(
                {
                    "path": record["relative_path"],
                    "source_mode": record["source_mode"],
                    "path_proves_train": record["path_proves_train"],
                    "size_bytes": record["size_bytes"],
                    "text_projection_sha256": record["text_projection_sha256"],
                    "rows_from_footer": record["rows_from_footer"],
                    "schema_names": record["schema_names"],
                    "requested_columns": requested,
                }
            )

        # Denylist locale : dev court materialise et vrai dev long. Seules les
        # colonnes texte/ID sont projetees ; les bytes audio restent intouches.
        heldout_files = []
        version_root = cache_root / "version=0"
        if version_root.is_dir():
            heldout_files.extend(
                path
                for path in version_root.rglob("*.parquet")
                if "split=dev" in path.relative_to(version_root).parts
            )
        eval_long_root = cache_root / "eval_long"
        if eval_long_root.is_dir():
            heldout_files.extend(eval_long_root.rglob("*.parquet"))
        heldout_files = sorted({Path(path) for path in heldout_files})
        assert heldout_files, (language, "aucun dev textuel local")
        heldout_plan = []
        heldout_inventory = []
        for path in heldout_files:
            path = require_child(path, cache_root)
            assert path.stat().st_size > 0 and not path.is_symlink()
            parquet_file = pq.ParquetFile(path, memory_map=False, pre_buffer=False)
            schema = parquet_file.schema_arrow
            text_column = choose_text_column(schema)
            id_column = choose_optional_column(
                schema,
                ("sample_id", "source_id", "original_id", "id"),
            )
            requested = [text_column]
            if id_column and id_column not in requested:
                requested.append(id_column)
            assert not any(is_audio_column(name) for name in requested)
            heldout_role = (
                "eval_long"
                if "eval_long" in path.relative_to(cache_root).parts
                else "dev"
            )
            projection_sha256, projection_rows = parquet_projection_sha256(
                path,
                requested,
            )
            record = {
                "path": str(path),
                "relative_path": str(path.relative_to(cache_root)),
                "size_bytes": path.stat().st_size,
                "text_projection_sha256": projection_sha256,
                "rows_from_footer": int(parquet_file.metadata.num_rows),
                "text_column": text_column,
                "id_column": id_column,
                "requested_columns": requested,
                "heldout_role": heldout_role,
                "audio_columns_requested": [],
            }
            assert projection_rows == record["rows_from_footer"]
            heldout_plan.append(record)
            heldout_inventory.append(
                {
                    key: record[key]
                    for key in (
                        "relative_path",
                        "size_bytes",
                        "text_projection_sha256",
                        "rows_from_footer",
                        "requested_columns",
                        "heldout_role",
                    )
                }
            )

        selected[language] = {
            "cache_root": str(cache_root),
            "cache_complete_sha256": row["complete_sha256"],
            "k1_snapshot": row,
            "train_parts_count": len(train_parts),
            "local_text_mode": local_text_mode,
            "local_text_plan": local_text_plan,
            "local_text_inventory_sha256": sha256_bytes(
                canonical_json(local_text_inventory).encode("utf-8")
            ),
            "local_text_rows_from_footers": sum(
                item["rows_from_footer"] for item in local_text_plan
            ),
            "heldout_text_plan": heldout_plan,
            "heldout_inventory_sha256": sha256_bytes(
                canonical_json(heldout_inventory).encode("utf-8")
            ),
            "heldout_rows_from_footers": sum(
                item["rows_from_footer"] for item in heldout_plan
            ),
        }
    assert set(selected) == set(LANGUAGES)
    return selected


SELECTED_CACHES = validate_selected_caches(K1_CACHE_SNAPSHOT)
print("Caches locaux selectionnes et sources texte auditees :")
for _language in LANGUAGES:
    _cache = SELECTED_CACHES[_language]
    print(
        " ",
        _language,
        "| mode=",
        _cache["local_text_mode"],
        "| parts=",
        len(_cache["local_text_plan"]),
        "| lignes footer=",
        _cache["local_text_rows_from_footers"],
    )


def locate_v4_manifest() -> dict[str, Any]:
    rows = [
        row
        for row in K1_REPORT["local_inventory"]
        if row.get("name") == "selected_records_v4"
    ]
    assert len(rows) == 1
    record = rows[0]
    path = require_child(record["path"], FT_ROOT)
    assert path.is_file() and path.suffix.lower() == ".parquet"
    if record.get("sha256"):
        assert sha256_file(path) == record["sha256"]
        digest = record["sha256"]
    else:
        print("Hash du manifest V4...")
        digest = sha256_file(path)
    parquet_file = pq.ParquetFile(path, memory_map=False, pre_buffer=False)
    schema = parquet_file.schema_arrow
    text_column = choose_text_column(schema)
    raw_text_column = choose_optional_column(schema, ("text", "text_raw"))
    text_norm_column = choose_optional_column(schema, ("text_norm",))
    language_column = choose_optional_column(schema, ("language", "lang"))
    split_column = choose_optional_column(schema, ("split",))
    id_column = choose_optional_column(schema, ("sample_id", "id"))
    assert language_column and split_column
    requested = []
    for column in (
        raw_text_column,
        text_norm_column,
        text_column,
        language_column,
        split_column,
        id_column,
    ):
        if column and column not in requested:
            requested.append(column)
    assert not any(is_audio_column(name) for name in requested)
    return {
        "path": str(path),
        "sha256": digest,
        "rows_from_footer": int(parquet_file.metadata.num_rows),
        "schema_names": list(schema.names),
        "raw_text_column": raw_text_column,
        "text_norm_column": text_norm_column,
        "text_column": text_column,
        "language_column": language_column,
        "split_column": split_column,
        "id_column": id_column,
        "requested_columns": requested,
        "audio_columns_requested": [],
    }


V4_MANIFEST = locate_v4_manifest()
print("Manifest V4 fige :", V4_MANIFEST["sha256"][:16])


# ---------------------------------------------------------------------------
# 4. Normalisation Unicode conservatrice et calibration V4
# ---------------------------------------------------------------------------

MOJIBAKE_RE = re.compile(
    r"(?:Ã.|Â.|Ä.|Å.|â[€‚„…†‡ˆ‰Š‹ŒŽ‘’“”•–—˜™š›œžŸ]|ï¿½)"
)
APOSTROPHE_DASH_TRANSLATION = str.maketrans(
    {
        "’": "'",
        "‘": "'",
        "ʼ": "'",
        "`": "'",
        "–": " ",
        "—": " ",
        "‐": " ",
        "‑": " ",
    }
)


def suspect_score(text: str) -> int:
    return (
        int(ftfy_badness(text))
        + 4 * text.count("\ufffd")
        + len(MOJIBAKE_RE.findall(text))
    )


def letter_ratio(text: str) -> float:
    letters_marks = sum(
        char.isalpha() or unicodedata.category(char).startswith("M")
        for char in text
    )
    return letters_marks / max(1, len(text))


def repair_text(raw: Any) -> tuple[str | None, str]:
    if raw is None:
        return None, "reject_null"
    text = unicodedata.normalize("NFC", str(raw))
    if "\ufffd" in text:
        return None, "reject_replacement_char"
    if not MOJIBAKE_RE.search(text):
        return text, "unchanged"
    candidate = unicodedata.normalize(
        "NFC",
        ftfy.fix_text(text, normalization="NFC"),
    )
    length_ratio = len(candidate) / max(1, len(text))
    safe = (
        "\ufffd" not in candidate
        and suspect_score(candidate) < suspect_score(text)
        and letter_ratio(candidate) + 0.02 >= letter_ratio(text)
        and 0.8 <= length_ratio <= 1.2
    )
    return (candidate, "fixed") if safe else (text, "suspect_unfixed")


def normalize_lm_text(raw: Any) -> tuple[str | None, str, list[str]]:
    repaired, mojibake_status = repair_text(raw)
    if repaired is None:
        return None, mojibake_status, []
    text = unicodedata.normalize(
        "NFC",
        repaired.translate(APOSTROPHE_DASH_TRANSLATION).casefold(),
    )
    text = uregex.sub(r"[^\p{L}\p{M}\p{N}'-]+", " ", text)
    text = uregex.sub(
        r"(?<![\p{L}\p{M}])['-]|['-](?![\p{L}\p{M}])",
        " ",
        text,
    )
    text = unicodedata.normalize("NFC", " ".join(text.split()))
    if not text:
        return None, "reject_empty", []
    flags = []
    if any(char.isdigit() for char in text):
        flags.append("contains_digit")
    if mojibake_status == "suspect_unfixed":
        flags.append("suspect_mojibake")
    if letter_ratio(text) < 0.45:
        return None, "reject_low_letter_ratio", flags
    return text, mojibake_status, flags


def chunk_normalized_text(text: str) -> list[str]:
    words = text.split()
    if len(words) <= MAX_WORDS_PER_LINE:
        return [text]
    chunks = [
        words[index : index + MAX_WORDS_PER_LINE]
        for index in range(0, len(words), MAX_WORDS_PER_LINE)
    ]
    if len(chunks) > 1 and len(chunks[-1]) < 8:
        needed = min(8 - len(chunks[-1]), len(chunks[-2]) - 1)
        if needed > 0:
            chunks[-1] = chunks[-2][-needed:] + chunks[-1]
            chunks[-2] = chunks[-2][:-needed]
    output = [" ".join(chunk) for chunk in chunks if chunk]
    assert output and all(
        1 <= len(chunk.split()) <= MAX_WORDS_PER_LINE for chunk in output
    )
    return output


def canonical_language(value: Any) -> str | None:
    normalized = str(value).strip().lower().replace("-", "_")
    base = normalized.split("_", 1)[0]
    for language, aliases in LANGUAGE_ALIASES.items():
        normalized_aliases = {alias.replace("-", "_") for alias in aliases}
        if (
            normalized == language
            or normalized in normalized_aliases
            or base == language
            or base in normalized_aliases
        ):
            return language
    return None


def calibrate_normalizer(max_rows: int = 10_000) -> dict[str, Any]:
    input_column = V4_MANIFEST["raw_text_column"] or V4_MANIFEST["text_column"]
    paired_reference = bool(
        V4_MANIFEST["text_norm_column"]
        and V4_MANIFEST["text_norm_column"] != input_column
    )
    reference_column = (
        V4_MANIFEST["text_norm_column"] if paired_reference else None
    )
    assert input_column and V4_MANIFEST["language_column"]
    reference_mode = "paired_legacy" if paired_reference else "self_consistency"
    rows_per_language = max(100, max_rows // len(LANGUAGES))

    def alnum_signature(value: Any) -> str | None:
        repaired, _ = repair_text(value)
        if repaired is None:
            return None
        folded = unicodedata.normalize("NFC", repaired.casefold())
        return "".join(
            char
            for char in folded
            if unicodedata.category(char)[0] in {"L", "M", "N"}
        )

    parquet_file = pq.ParquetFile(
        V4_MANIFEST["path"],
        memory_map=False,
        pre_buffer=False,
    )
    columns = []
    for column in (
        input_column,
        reference_column,
        V4_MANIFEST["language_column"],
        V4_MANIFEST["split_column"],
    ):
        if column and column not in columns:
            columns.append(column)
    counts = {language: Counter() for language in LANGUAGES}
    for batch in parquet_file.iter_batches(
        batch_size=4096,
        columns=columns,
        use_threads=False,
    ):
        values = batch.to_pydict()
        for row_index in range(batch.num_rows):
            split = values[V4_MANIFEST["split_column"]][row_index]
            if str(split).strip().lower() != "train":
                continue
            language = canonical_language(
                values[V4_MANIFEST["language_column"]][row_index]
            )
            if language not in counts:
                continue
            stats = counts[language]
            if stats["attempted"] >= rows_per_language:
                continue
            raw = values[input_column][row_index]
            expected = values[reference_column][row_index] if reference_column else None
            if paired_reference and expected is None:
                continue
            stats["attempted"] += 1
            produced, status, _ = normalize_lm_text(raw)
            if produced is None:
                stats["rejected"] += 1
            else:
                stats["accepted"] += 1
                stats[f"status|{status}"] += 1
                source_nfc = unicodedata.normalize("NFC", str(raw)).strip()
                stats["changed"] += int(produced != source_nfc)
                if paired_reference:
                    expected_nfc = unicodedata.normalize(
                        "NFC", str(expected)
                    ).strip()
                    stats["exact_legacy"] += int(produced == expected_nfc)
                produced_again, _, _ = normalize_lm_text(produced)
                stats["idempotent"] += int(produced_again == produced)
                stats["nfc"] += int(unicodedata.is_normalized("NFC", produced))
                stats["replacement_characters"] += produced.count("\ufffd")
                stats["alnum_preserved"] += int(
                    alnum_signature(raw) == alnum_signature(produced)
                )
        if all(
            counts[language]["attempted"] >= rows_per_language
            for language in LANGUAGES
        ):
            break

    per_language: dict[str, Any] = {}
    for language in LANGUAGES:
        stats = counts[language]
        attempted = stats["attempted"]
        accepted = stats["accepted"]
        assert attempted >= rows_per_language, (language, attempted, rows_per_language)
        assert accepted > 0, language
        acceptance_rate = accepted / attempted
        idempotence_rate = stats["idempotent"] / accepted
        alnum_preservation_rate = stats["alnum_preserved"] / accepted
        nfc_rate = stats["nfc"] / accepted
        rejection_rate = stats["rejected"] / attempted
        exact_rate = (
            stats["exact_legacy"] / accepted if paired_reference else None
        )
        assert rejection_rate <= NORMALIZATION_REJECTION_MAX, (
            "NORMALIZER_REJECTION_RATE_TOO_HIGH",
            language,
            rejection_rate,
        )
        assert idempotence_rate >= NORMALIZATION_IDEMPOTENCE_MIN, (
            "NORMALIZER_NOT_IDEMPOTENT",
            language,
            idempotence_rate,
        )
        assert alnum_preservation_rate == 1.0, (
            "NORMALIZER_ALNUM_CHANGED",
            language,
            alnum_preservation_rate,
        )
        assert nfc_rate == 1.0 and stats["replacement_characters"] == 0
        if paired_reference:
            assert exact_rate is not None
            assert exact_rate >= NORMALIZATION_COMPATIBILITY_MIN, (
                "NORMALIZER_INCOMPATIBLE_WITH_V4",
                language,
                exact_rate,
            )
        per_language[language] = {
            "attempted": attempted,
            "accepted": accepted,
            "rejected": stats["rejected"],
            "acceptance_rate": acceptance_rate,
            "exact_rate": exact_rate,
            "idempotence_rate": idempotence_rate,
            "alnum_preservation_rate": alnum_preservation_rate,
            "nfc_rate": nfc_rate,
            "changed_rate": stats["changed"] / accepted,
            "fixed": stats["status|fixed"],
            "suspect_unfixed": stats["status|suspect_unfixed"],
            "replacement_characters": stats["replacement_characters"],
        }

    attempted = sum(item["attempted"] for item in per_language.values())
    accepted = sum(item["accepted"] for item in per_language.values())
    acceptance_rate = accepted / attempted
    idempotence_rate = sum(
        counts[language]["idempotent"] for language in LANGUAGES
    ) / accepted
    alnum_preservation_rate = sum(
        counts[language]["alnum_preserved"] for language in LANGUAGES
    ) / accepted
    nfc_rate = sum(counts[language]["nfc"] for language in LANGUAGES) / accepted
    exact_rate = (
        sum(counts[language]["exact_legacy"] for language in LANGUAGES)
        / accepted
        if paired_reference
        else None
    )
    compatibility_score = (
        exact_rate
        if exact_rate is not None
        else min(
            acceptance_rate,
            idempotence_rate,
            alnum_preservation_rate,
            nfc_rate,
        )
    )
    assert compatibility_score >= NORMALIZATION_COMPATIBILITY_MIN
    return {
        "mode": reference_mode,
        "input_column": input_column,
        "reference_column": reference_column,
        "rows_per_language_target": rows_per_language,
        "rows_attempted": attempted,
        "rows_accepted": accepted,
        "rows_rejected": attempted - accepted,
        "acceptance_rate": acceptance_rate,
        "exact_rate": exact_rate,
        "idempotence_rate": idempotence_rate,
        "alnum_preservation_rate": alnum_preservation_rate,
        "nfc_rate": nfc_rate,
        "replacement_characters": sum(
            counts[language]["replacement_characters"] for language in LANGUAGES
        ),
        "compatibility_score": compatibility_score,
        "per_language": per_language,
        "minimum_compatibility_score": NORMALIZATION_COMPATIBILITY_MIN,
        "maximum_rejection_rate": NORMALIZATION_REJECTION_MAX,
        "minimum_idempotence_rate": NORMALIZATION_IDEMPOTENCE_MIN,
        "raw_text_written_to_report": False,
    }


NORMALIZER_CALIBRATION = calibrate_normalizer()
print(
    "Compatibilite normaliseur V4 :",
    round(NORMALIZER_CALIBRATION["compatibility_score"], 4),
    "| mode=",
    NORMALIZER_CALIBRATION["mode"],
    "| idempotence=",
    round(NORMALIZER_CALIBRATION["idempotence_rate"], 4),
)


# ---------------------------------------------------------------------------
# 5. Sources distantes text-only autorisees par 20.K1
# ---------------------------------------------------------------------------

EXPECTED_REMOTE_BY_LANGUAGE = {
    "sw": {"fleurs", "kenspeech_text"},
    "kik": set(),
    "kln": set(),
    "luo": {"fleurs", "kenpos_dholuo"},
    "som": {"fleurs"},
    "mas": set(),
}


def validate_remote_contract() -> tuple[
    HfApi,
    HfFileSystem,
    dict[str, dict[str, Any]],
]:
    assert HF_TOKEN, "Ajouter HF_TOKEN read-only aux Secrets Colab."
    api = HfApi(token=HF_TOKEN)
    filesystem = HfFileSystem(token=HF_TOKEN)
    api.whoami(token=HF_TOKEN)

    coverage = {
        row["language"]: set(row.get("remote_source_ids", []))
        for row in K1_REPORT["coverage"]
    }
    assert set(coverage) == set(LANGUAGES)
    for language in LANGUAGES:
        assert coverage[language] == EXPECTED_REMOTE_BY_LANGUAGE[language], (
            "REMOTE_SOURCE_SET_CHANGED",
            language,
            coverage[language],
            EXPECTED_REMOTE_BY_LANGUAGE[language],
        )

    inventory_by_id = {
        row["source_id"]: row for row in K1_REPORT["hf_inventory"]
    }
    remote_plan: dict[str, dict[str, Any]] = {}
    expected_paths = {
        ("fleurs", "sw"): "data/sw_ke/train.tsv",
        ("fleurs", "luo"): "data/luo_ke/train.tsv",
        ("fleurs", "som"): "data/so_so/train.tsv",
        ("kenspeech_text", "sw"): "transcripts_only.csv",
        ("kenpos_dholuo", "luo"): "dho/train.parquet",
    }

    for language in LANGUAGES:
        for source_id in sorted(coverage[language]):
            assert source_id in REMOTE_SOURCE_RULES
            source_record = inventory_by_id[source_id]
            rule = REMOTE_SOURCE_RULES[source_id]
            assert source_record["repo_id"] == rule["repo_id"]
            assert source_record["access_status"] == "OK"
            assert source_record["hub_read_access"] is True
            assert source_record["reported_private"] is False
            assert str(source_record.get("reported_gated", "")).strip().casefold() not in {
                "true",
                "1",
                "yes",
            }
            assert source_record["license_status"] == "APPROVED"
            assert source_record["license_matches_expected"] is not False
            assert source_record["provenance_status"] == "approved"
            assert source_record["ingest_policy"] == "extract"
            revision = str(source_record["resolved_revision"])
            assert re.fullmatch(r"[0-9a-fA-F]{40}", revision), revision
            repo_id = source_record["repo_id"]
            info = api.dataset_info(
                repo_id=repo_id,
                revision=revision,
                files_metadata=False,
                token=HF_TOKEN,
                timeout=120,
            )
            assert str(info.sha).lower() == revision.lower(), (info.sha, revision)

            path = expected_paths[(source_id, language)]
            assert not is_audio_column(path)
            tree_evidence = source_record.get("pinned_tree_evidence", {}).get(
                language,
                {},
            )
            assert tree_evidence.get("ok") is True, (source_id, language)
            assert tree_evidence.get("resolved_revision") == revision
            preview_contains_path = path in tree_evidence.get(
                "matching_paths_preview", []
            )
            path_infos = api.get_paths_info(
                repo_id=repo_id,
                paths=[path],
                repo_type="dataset",
                revision=revision,
                token=HF_TOKEN,
            )
            assert len(path_infos) == 1, (repo_id, revision, path)
            path_info = path_infos[0]
            remote_size = getattr(path_info, "size", None)
            assert remote_size is not None and int(remote_size) > 0
            lfs = getattr(path_info, "lfs", None)
            if hasattr(lfs, "__dict__"):
                lfs = dict(vars(lfs))
            elif lfs is not None and not isinstance(lfs, dict):
                lfs = {"value": str(lfs)}
            remote_identity = {
                "blob_id": getattr(path_info, "blob_id", None),
                "lfs": lfs,
                "xet_hash": getattr(path_info, "xet_hash", None),
            }
            assert any(value for value in remote_identity.values()), (
                "REMOTE_BLOB_IDENTITY_MISSING",
                repo_id,
                revision,
                path,
            )
            uri = f"hf://datasets/{repo_id}@{revision}/{path}"
            resolved = filesystem.resolve_path(uri)
            assert str(resolved.revision).lower() == revision.lower()
            key = f"{source_id}|{language}"
            remote_plan[key] = {
                "source_id": source_id,
                "language": language,
                "repo_id": repo_id,
                "revision": revision.lower(),
                "path": path,
                "uri": uri,
                "size_bytes": int(remote_size) if remote_size is not None else None,
                "remote_identity": remote_identity,
                "k1_preview_contains_path": preview_contains_path,
                "mode": rule["languages"][language]["mode"],
                "family": rule["family"],
                "priority": rule["priority"],
                "reported_licenses": source_record.get("reported_licenses", []),
                "license_status": source_record["license_status"],
                "audio_paths_opened": [],
            }
    assert set(remote_plan) == {
        "fleurs|sw",
        "kenspeech_text|sw",
        "fleurs|luo",
        "kenpos_dholuo|luo",
        "fleurs|som",
    }
    return api, filesystem, remote_plan


HF_API, HF_FS, REMOTE_PLAN = validate_remote_contract()
print("Sources distantes text-only figees :", len(REMOTE_PLAN))


# ---------------------------------------------------------------------------
# 6. Contrat immuable 20.K2 et chemins de reprise
# ---------------------------------------------------------------------------

NORMALIZER_CONTRACT = {
    "version": NORMALIZER_VERSION,
    "unicode_normalization": "NFC",
    "case": "casefold",
    "preserve": "letters_marks_numbers_internal_apostrophe_hyphen",
    "safe_ftfy_only": True,
    "compatibility_min": NORMALIZATION_COMPATIBILITY_MIN,
    "max_words_per_line": MAX_WORDS_PER_LINE,
    "near_min_words": MIN_NEAR_WORDS,
    "near_min_chars": MIN_NEAR_CHARS,
    "minhash_components": MINHASH_COMPONENTS,
    "bands": MINHASH_BANDS,
    "short_edit": NEAR_SHORT_EDIT,
    "short_jaccard": NEAR_SHORT_JACCARD,
    "long_edit": NEAR_LONG_EDIT,
    "long_jaccard": NEAR_LONG_JACCARD,
    "heldout_edit": HELDOUT_EDIT,
    "heldout_jaccard": HELDOUT_JACCARD,
}

PACKAGE_VERSIONS = {
    package: importlib.metadata.version(package)
    for package in (
        "pyarrow",
        "numpy",
        "pandas",
        "huggingface_hub",
        "ftfy",
        "regex",
        "rapidfuzz",
        "duckdb",
    )
}

K2_CONTRACT = {
    "schema": 1,
    "cell": "20.K2",
    "pipeline_revision": PIPELINE_REVISION,
    "runtime_requirements": {
        "python": ">=3.10",
        "packages": REQUIRED_MODULES,
    },
    "k1_run_id": K1_LATEST["run_id"],
    "k1_report_sha256": EXPECTED_K1_REPORT_SHA256,
    "k1_contract_sha256": K1_LATEST["contract_sha256"],
    "v4_manifest_sha256": V4_MANIFEST["sha256"],
    "normalizer": NORMALIZER_CONTRACT,
    "normalizer_calibration": NORMALIZER_CALIBRATION,
    "local_sources": {
        language: {
            "cache_root": cache["cache_root"],
            "cache_complete_sha256": cache["cache_complete_sha256"],
            "local_text_mode": cache["local_text_mode"],
            "local_text_inventory_sha256": cache[
                "local_text_inventory_sha256"
            ],
            "local_text_rows_from_footers": cache[
                "local_text_rows_from_footers"
            ],
            "heldout_inventory_sha256": cache["heldout_inventory_sha256"],
            "heldout_rows_from_footers": cache["heldout_rows_from_footers"],
        }
        for language, cache in SELECTED_CACHES.items()
    },
    "remote_sources": REMOTE_PLAN,
    "source_policy": {
        "train_only": True,
        "catalog_only_excluded": True,
        "review_excluded": True,
        "unknown_license_remote_excluded": True,
        "anv_remote_and_mirrors_excluded": True,
        "sw_local_scope": (
            "materialized_split_train_only_when_lm_text_is_absent"
        ),
        "audio_columns_requested": [],
        "raw_text_written_to_reports": False,
    },
}
K2_CONTRACT_SHA256 = sha256_bytes(canonical_json(K2_CONTRACT).encode("utf-8"))
K2_RUN_ID = K2_CONTRACT_SHA256[:16]
FINAL_RUN_DIR = CORPUS_ROOT / K2_RUN_ID


def validate_completed_run(path: Path) -> dict[str, Any] | None:
    try:
        complete = read_small_json(path / "_COMPLETE.json")
        assert complete["cell"] == "20.K2"
        assert complete["run_id"] == K2_RUN_ID
        assert complete["contract_sha256"] == K2_CONTRACT_SHA256
        assert complete["status"] == "PASS_READY_FOR_20_K3_KENLM_BUILD"
        required_artifacts = {
            "contract.json",
            "kenlm_text_corpus_20_K2.json",
            "language_summary_20_K2.csv",
            *{f"final/{language}.parquet" for language in LANGUAGES},
            *{f"final/train_{language}.txt.gz" for language in LANGUAGES},
            *{f"manifests/candidate_{language}.json" for language in LANGUAGES},
            *{f"manifests/dedup_{language}.json" for language in LANGUAGES},
        }
        assert set(complete["artifacts"]) == required_artifacts
        for relative, metadata in complete["artifacts"].items():
            artifact = require_child(path / relative, path)
            assert artifact.is_file()
            assert artifact.stat().st_size == metadata["bytes"]
            assert sha256_file(artifact) == metadata["sha256"]
        return complete
    except Exception:
        return None


RUN_ALREADY_COMPLETE = False
if FINAL_RUN_DIR.exists():
    _existing_complete = validate_completed_run(FINAL_RUN_DIR)
    assert _existing_complete is not None, (
        "FINAL_RUN_DIR_EXISTE_MAIS_EST_INVALIDE",
        FINAL_RUN_DIR,
    )
    RUN_ALREADY_COMPLETE = True
    STAGING_RUN_DIR = FINAL_RUN_DIR
else:
    STAGING_RUN_DIR = CORPUS_ROOT / f"{K2_RUN_ID}.staging"
LOCAL_WORK_DIR = Path("/content") / f"kenlm_v6_20_K2_{K2_RUN_ID}"
for directory in (
    STAGING_RUN_DIR / "candidates",
    STAGING_RUN_DIR / "final",
    STAGING_RUN_DIR / "manifests",
    LOCAL_WORK_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

atomic_json(K2_CONTRACT, STAGING_RUN_DIR / "contract.json")
print("Contrat 20.K2 :", K2_CONTRACT_SHA256)
print(
    "Run Drive      :",
    STAGING_RUN_DIR,
    "| deja complet =",
    RUN_ALREADY_COMPLETE,
)


# ---------------------------------------------------------------------------
# 7. Extraction/normalisation des candidats, reprise par langue
# ---------------------------------------------------------------------------


class CandidateSink:
    def __init__(self, path: Path, language: str, batch_rows: int = 5000):
        self.path = Path(path)
        self.language = language
        self.batch_rows = batch_rows
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.writer = pq.ParquetWriter(
            self.path,
            CANDIDATE_SCHEMA,
            compression="zstd",
            use_dictionary=True,
            write_statistics=True,
        )
        self.buffer: list[dict[str, Any]] = []
        self.accepted = 0
        self.parent_rows = 0
        self.rejections: Counter[str] = Counter()
        self.mojibake: Counter[str] = Counter()
        self.source_rows: Counter[str] = Counter()
        self.source_segments: Counter[str] = Counter()

    def reject(self, reason: str, source_id: str) -> None:
        self.rejections[reason] += 1
        self.source_rows[f"{source_id}|rejected"] += 1

    def add(
        self,
        *,
        raw_text: Any,
        source_id: str,
        source_family: str,
        source_revision: str,
        source_path: str,
        source_row_id: str,
        source_priority: int,
    ) -> None:
        self.parent_rows += 1
        self.source_rows[f"{source_id}|input"] += 1
        normalized, mojibake_status, flags = normalize_lm_text(raw_text)
        self.mojibake[mojibake_status] += 1
        if normalized is None:
            self.reject(mojibake_status, source_id)
            return
        chunks = chunk_normalized_text(normalized)
        for segment_index, text in enumerate(chunks):
            word_count = len(text.split())
            if word_count < 1:
                self.reject("reject_no_word", source_id)
                continue
            exact_sha = sha256_bytes(
                f"{self.language}\0{text}".encode("utf-8")
            )
            record_id = sha256_bytes(
                (
                    f"{self.language}\0{source_id}\0{source_revision}\0"
                    f"{source_path}\0{source_row_id}\0{segment_index}\0{exact_sha}"
                ).encode("utf-8")
            )[:32]
            record = {
                "record_id": record_id,
                "language": self.language,
                "source_id": source_id,
                "source_family": source_family,
                "source_revision": source_revision,
                "source_split": "train",
                "source_path": source_path,
                "source_row_id": source_row_id,
                "source_priority": int(source_priority),
                "segment_index": int(segment_index),
                "text_norm": text,
                "exact_sha256": exact_sha,
                "word_count": int(word_count),
                "char_count": int(len(text)),
                "mojibake_status": mojibake_status,
                "contains_digit": any(char.isdigit() for char in text),
                "quality_flags": ",".join(sorted(flags)),
            }
            self.buffer.append(record)
            self.accepted += 1
            self.source_segments[source_id] += 1
            if len(self.buffer) >= self.batch_rows:
                self.flush()

    def flush(self) -> None:
        if not self.buffer:
            return
        table = pa.Table.from_pylist(self.buffer, schema=CANDIDATE_SCHEMA)
        self.writer.write_table(table, row_group_size=len(self.buffer))
        self.buffer.clear()

    def close(self) -> dict[str, Any]:
        self.flush()
        self.writer.close()
        metadata = pq.read_metadata(self.path)
        assert metadata.num_rows == self.accepted
        return {
            "candidate_rows": self.accepted,
            "parent_rows": self.parent_rows,
            "rejections": dict(sorted(self.rejections.items())),
            "mojibake": dict(sorted(self.mojibake.items())),
            "source_rows": dict(sorted(self.source_rows.items())),
            "source_segments": dict(sorted(self.source_segments.items())),
        }


def normalize_split(value: Any) -> str:
    return str(value).strip().lower().replace("-", "_")


def parse_boolean_strict(value: Any, *, field: str, context: str) -> bool:
    """Parse un drapeau sans accepter le piege bool("False") == True."""
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        if value == 1:
            return True
        if value == 0:
            return False
    normalized = str(value).strip().casefold() if value is not None else ""
    if normalized in {"true", "1", "yes", "y", "oui", "dev", "valid", "validation"}:
        return True
    if normalized in {"false", "0", "no", "n", "non", "train"}:
        return False
    raise AssertionError(
        f"Valeur booleenne ambigue pour {field} ({context}) : {value!r}"
    )


def iter_local_text_rows(
    language: str,
    cache: dict[str, Any],
) -> Iterator[dict[str, Any]]:
    cache_root = Path(cache["cache_root"])
    plan = cache["local_text_plan"]
    for file_index, record in enumerate(plan, 1):
        if record["source_mode"] == "lm_text":
            allowed_root = cache_root / "lm_text"
        elif record["source_mode"] == "train_parquet_projection":
            allowed_root = cache_root
            assert record["path_proves_train"] is True
        else:
            raise AssertionError(record["source_mode"])
        path = require_child(record["path"], allowed_root)
        parquet_file = pq.ParquetFile(path, memory_map=False, pre_buffer=False)
        columns = record["requested_columns"]
        assert not any(is_audio_column(name) for name in columns)
        rows_seen = 0
        row_offset = 0
        for batch in parquet_file.iter_batches(
            batch_size=8192,
            columns=columns,
            use_threads=False,
        ):
            assert batch.schema.names == columns
            values = batch.to_pydict()
            for local_index in range(batch.num_rows):
                rows_seen += 1
                current_row = row_offset + local_index
                if record["is_dev_column"]:
                    is_dev = values[record["is_dev_column"]][local_index]
                    if parse_boolean_strict(
                        is_dev,
                        field=record["is_dev_column"],
                        context=f"{record['relative_path']}:{current_row}",
                    ):
                        continue
                if record["split_column"]:
                    split = normalize_split(
                        values[record["split_column"]][local_index]
                    )
                    if split != "train":
                        continue
                else:
                    assert record["path_proves_train"] is True or record[
                        "is_dev_column"
                    ]
                if record["language_column"]:
                    declared = canonical_language(
                        values[record["language_column"]][local_index]
                    )
                    if declared != language:
                        continue
                raw_id = (
                    values[record["id_column"]][local_index]
                    if record["id_column"]
                    else current_row
                )
                yield {
                    "raw_text": values[record["text_column"]][local_index],
                    "source_row_id": str(raw_id),
                    "source_path": record["relative_path"],
                }
            row_offset += batch.num_rows
        assert rows_seen == record["rows_from_footer"], (
            path,
            rows_seen,
            record["rows_from_footer"],
        )
        if file_index % 20 == 0 or file_index == len(plan):
            print(
                f"  {language} {cache['local_text_mode']} "
                f"{file_index}/{len(plan)}"
            )


def add_v4_train(language: str, sink: CandidateSink) -> dict[str, int]:
    parquet_file = pq.ParquetFile(
        V4_MANIFEST["path"],
        memory_map=False,
        pre_buffer=False,
    )
    columns = V4_MANIFEST["requested_columns"]
    input_rows = 0
    selected_rows = 0
    row_offset = 0
    preferred_text = V4_MANIFEST["text_norm_column"] or V4_MANIFEST["text_column"]
    for batch in parquet_file.iter_batches(
        batch_size=8192,
        columns=columns,
        use_threads=False,
    ):
        values = batch.to_pydict()
        for index in range(batch.num_rows):
            input_rows += 1
            if normalize_split(values[V4_MANIFEST["split_column"]][index]) != "train":
                continue
            if canonical_language(values[V4_MANIFEST["language_column"]][index]) != language:
                continue
            selected_rows += 1
            raw_id = (
                values[V4_MANIFEST["id_column"]][index]
                if V4_MANIFEST["id_column"]
                else row_offset + index
            )
            sink.add(
                raw_text=values[preferred_text][index],
                source_id="local_v4_train",
                source_family="afrivoices_v4",
                source_revision=V4_MANIFEST["sha256"],
                source_path=Path(V4_MANIFEST["path"]).name,
                source_row_id=str(raw_id),
                source_priority=20,
            )
        row_offset += batch.num_rows
    assert input_rows == V4_MANIFEST["rows_from_footer"]
    return {"input_rows": input_rows, "selected_rows": selected_rows}


class HashingRawReader(io.RawIOBase):
    """Lecteur sequentiel qui mesure et empreinte sans conserver le fichier."""

    def __init__(self, source: Any):
        super().__init__()
        self.source = source
        self.digest = hashlib.sha256()
        self.bytes_read = 0

    def readable(self) -> bool:
        return True

    def readinto(self, buffer: Any) -> int:
        data = self.source.read(len(buffer))
        if not data:
            return 0
        size = len(data)
        buffer[:size] = data
        self.digest.update(data)
        self.bytes_read += size
        return size

    def hexdigest(self) -> str:
        return self.digest.hexdigest()


def validate_remote_stream(record: dict[str, Any], meter: HashingRawReader) -> None:
    assert meter.bytes_read == record["size_bytes"], (
        "REMOTE_STREAM_SIZE_MISMATCH",
        record["path"],
        meter.bytes_read,
        record["size_bytes"],
    )


def add_fleurs(record: dict[str, Any], sink: CandidateSink) -> dict[str, Any]:
    resolved = HF_FS.resolve_path(record["uri"])
    assert str(resolved.revision).lower() == record["revision"].lower()
    rows = 0
    with HF_FS.open(
        record["uri"], "rb", block_size=64 * 1024, cache_type="none"
    ) as remote_handle:
        meter = HashingRawReader(remote_handle)
        with io.TextIOWrapper(
            io.BufferedReader(meter, buffer_size=64 * 1024),
            encoding="utf-8-sig",
            errors="strict",
            newline="",
        ) as text_handle:
            reader = csv.reader(text_handle, delimiter="\t", quoting=csv.QUOTE_NONE)
            for physical_line, row in enumerate(reader, 1):
                if not row:
                    continue
                assert len(row) == 7, (record["path"], physical_line, len(row))
                int(row[0])
                int(row[5])
                assert row[6].strip().upper() in {"MALE", "FEMALE", "OTHER"}
                sink.add(
                    raw_text=row[3],
                    source_id="fleurs",
                    source_family=record["family"],
                    source_revision=record["revision"],
                    source_path=record["path"],
                    source_row_id=f"line:{physical_line}:id:{row[0]}",
                    source_priority=record["priority"],
                )
                rows += 1
        validate_remote_stream(record, meter)
    assert rows > 0
    return {
        "rows": rows,
        "file_sha256": meter.hexdigest(),
        "parser_bytes": meter.bytes_read,
        "requested_columns": ["transcription"],
        "audio_columns_requested": [],
    }


def add_kenspeech(record: dict[str, Any], sink: CandidateSink) -> dict[str, Any]:
    rows = 0
    resolved = HF_FS.resolve_path(record["uri"])
    assert str(resolved.revision).lower() == record["revision"].lower()
    with HF_FS.open(
        record["uri"], "rb", block_size=64 * 1024, cache_type="none"
    ) as remote_handle:
        meter = HashingRawReader(remote_handle)
        with io.TextIOWrapper(
            io.BufferedReader(meter, buffer_size=64 * 1024),
            encoding="utf-8-sig",
            errors="strict",
            newline="",
        ) as text_handle:
            reader = csv.reader(text_handle)
            first = next(reader, None)
            assert first, "KENSPEECH_EMPTY"
            header = [value.strip().lower() for value in first]
            text_index = next(
                (
                    header.index(candidate)
                    for candidate in ("transcript", "text", "transcription")
                    if candidate in header
                ),
                None,
            )
            if text_index is None:
                assert len(first) == 1, "FAIL_SCHEMA_AMBIGUOUS_KENSPEECH"
                text_index = 0
                pending = [(1, first)]
            else:
                pending = []

            def consume(physical_line: int, row: list[str]) -> None:
                nonlocal rows
                if not row:
                    return
                assert text_index is not None and text_index < len(row)
                sink.add(
                    raw_text=row[text_index],
                    source_id="kenspeech_text",
                    source_family=record["family"],
                    source_revision=record["revision"],
                    source_path=record["path"],
                    source_row_id=f"line:{physical_line}",
                    source_priority=record["priority"],
                )
                rows += 1

            for physical_line, row in pending:
                consume(physical_line, row)
            for physical_line, row in enumerate(reader, 2):
                consume(physical_line, row)
        validate_remote_stream(record, meter)
    assert rows > 0
    return {
        "rows": rows,
        "file_sha256": meter.hexdigest(),
        "parser_bytes": meter.bytes_read,
        "requested_columns": ["transcript"],
        "audio_columns_requested": [],
    }


def add_kenpos(record: dict[str, Any], sink: CandidateSink) -> dict[str, Any]:
    resolved = HF_FS.resolve_path(record["uri"])
    assert str(resolved.revision).lower() == record["revision"].lower()
    with HF_FS.open(
        record["uri"],
        "rb",
        block_size=256 * 1024,
        cache_type="none",
    ) as handle:
        parquet_file = pq.ParquetFile(handle, pre_buffer=False, buffer_size=0)
        required = ["token", "filename", "sentence_id", "position"]
        assert set(required).issubset(parquet_file.schema_arrow.names), (
            record["path"],
            parquet_file.schema_arrow.names,
        )
        assert not any(is_audio_column(name) for name in required)
        groups: dict[tuple[str, str], list[tuple[int, str]]] = defaultdict(list)
        physical_rows = 0
        nonnull_token_rows = 0
        for batch in parquet_file.iter_batches(
            batch_size=8192,
            columns=required,
            use_threads=False,
        ):
            assert batch.schema.names == required
            physical_rows += batch.num_rows
            values = batch.to_pydict()
            for index in range(batch.num_rows):
                token = values["token"][index]
                if token is None:
                    continue
                assert values["filename"][index] is not None
                assert values["sentence_id"][index] is not None
                assert values["position"][index] is not None
                key = (
                    str(values["filename"][index]),
                    str(values["sentence_id"][index]),
                )
                position = int(values["position"][index])
                groups[key].append((position, str(token)))
                nonnull_token_rows += 1
        parquet_rows = int(parquet_file.metadata.num_rows)
        assert physical_rows == parquet_rows
    sentences = 0
    for key in sorted(groups):
        ordered = sorted(groups[key], key=lambda item: (item[0], item[1]))
        positions = [item[0] for item in ordered]
        assert len(positions) == len(set(positions)), (key, "positions dupliquees")
        text = " ".join(item[1] for item in ordered)
        sink.add(
            raw_text=text,
            source_id="kenpos_dholuo",
            source_family=record["family"],
            source_revision=record["revision"],
            source_path=record["path"],
            source_row_id=f"{key[0]}:{key[1]}",
            source_priority=record["priority"],
        )
        sentences += 1
    assert sentences > 0
    return {
        "parquet_num_rows": parquet_rows,
        "physical_rows": physical_rows,
        "nonnull_token_rows": nonnull_token_rows,
        "sentences": sentences,
        "requested_columns": required,
        "audio_columns_requested": [],
        "transport_bytes_metered": False,
    }


def valid_checkpoint(
    manifest_path: Path,
    data_path: Path,
    phase: str,
    language: str,
) -> bool:
    try:
        manifest = read_small_json(manifest_path)
        schema = pq.ParquetFile(
            data_path,
            memory_map=False,
            pre_buffer=False,
        ).schema_arrow
        return bool(
            manifest.get("phase") == phase
            and manifest.get("language") == language
            and manifest.get("contract_sha256") == K2_CONTRACT_SHA256
            and data_path.is_file()
            and data_path.stat().st_size == manifest.get("bytes")
            and sha256_file(data_path) == manifest.get("sha256")
            and pq.read_metadata(data_path).num_rows == manifest.get("rows")
            and schema.equals(CANDIDATE_SCHEMA, check_metadata=False)
        )
    except Exception:
        return False


def publish_file(local_path: Path, drive_path: Path) -> None:
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = drive_path.with_name(drive_path.name + f".tmp-{os.getpid()}")
    shutil.copy2(local_path, temporary)
    assert sha256_file(temporary) == sha256_file(local_path)
    os.replace(temporary, drive_path)


def extract_language_candidates(language: str) -> tuple[Path, dict[str, Any]]:
    drive_path = STAGING_RUN_DIR / "candidates" / f"{language}.parquet"
    manifest_path = STAGING_RUN_DIR / "manifests" / f"candidate_{language}.json"
    local_path = LOCAL_WORK_DIR / f"candidate_{language}.parquet"
    if valid_checkpoint(
        manifest_path,
        drive_path,
        "candidate_extraction",
        language,
    ):
        print(f"[reprise] candidats {language}")
        shutil.copy2(drive_path, local_path)
        assert sha256_file(local_path) == read_small_json(manifest_path)["sha256"]
        return local_path, read_small_json(manifest_path)

    if local_path.exists():
        local_path.unlink()
    sink = CandidateSink(local_path, language)
    source_audits: dict[str, Any] = {}

    print(
        f"\n=== EXTRACTION {language} : "
        f"{SELECTED_CACHES[language]['local_text_mode']} local ==="
    )
    local_mode = SELECTED_CACHES[language]["local_text_mode"]
    if local_mode == "lm_text":
        local_source_id = f"local_v5_full_{language}"
        local_source_family = "afrivoices_v5_local_full"
    else:
        assert language == "sw"
        local_source_id = "local_sw_materialized_train"
        local_source_family = "afrivoices_sw_materialized_train"
        print(
            "  AVERTISSEMENT couverture sw : seuls les Parquets train "
            "materialises et figes par 20.K1 sont inclus."
        )
    local_rows = 0
    for record in iter_local_text_rows(language, SELECTED_CACHES[language]):
        sink.add(
            raw_text=record["raw_text"],
            source_id=local_source_id,
            source_family=local_source_family,
            source_revision=SELECTED_CACHES[language]["cache_complete_sha256"],
            source_path=record["source_path"],
            source_row_id=record["source_row_id"],
            source_priority=10,
        )
        local_rows += 1
    assert local_rows > 0
    source_audits["local_cache_text"] = {
        "train_rows": local_rows,
        "mode": local_mode,
        "source_id": local_source_id,
        "audio_columns_requested": [],
    }

    print(f"=== EXTRACTION {language} : replay V4 train ===")
    source_audits["local_v4_train"] = add_v4_train(language, sink)

    for key in sorted(REMOTE_PLAN):
        remote = REMOTE_PLAN[key]
        if remote["language"] != language:
            continue
        print("=== EXTRACTION DISTANTE", key, "===")
        if remote["source_id"] == "fleurs":
            audit = add_fleurs(remote, sink)
        elif remote["source_id"] == "kenspeech_text":
            audit = add_kenspeech(remote, sink)
        elif remote["source_id"] == "kenpos_dholuo":
            audit = add_kenpos(remote, sink)
        else:
            raise AssertionError(remote["source_id"])
        source_audits[key] = audit

    sink_stats = sink.close()
    assert sink_stats["candidate_rows"] > 0
    digest = sha256_file(local_path)
    publish_file(local_path, drive_path)
    manifest = {
        "schema": 1,
        "phase": "candidate_extraction",
        "language": language,
        "contract_sha256": K2_CONTRACT_SHA256,
        "relative_path": f"candidates/{language}.parquet",
        "bytes": drive_path.stat().st_size,
        "sha256": digest,
        "rows": int(pq.read_metadata(drive_path).num_rows),
        "sink_stats": sink_stats,
        "source_audits": source_audits,
        "audio_columns_requested": [],
        "audio_bytes_read": 0,
    }
    atomic_json(manifest, manifest_path)
    return local_path, manifest


CANDIDATE_PATHS: dict[str, Path] = {}
CANDIDATE_MANIFESTS: dict[str, dict[str, Any]] = {}
if not RUN_ALREADY_COMPLETE:
    for _language in LANGUAGES:
        _path, _manifest = extract_language_candidates(_language)
        CANDIDATE_PATHS[_language] = _path
        CANDIDATE_MANIFESTS[_language] = _manifest
        print(
            "Candidats",
            _language,
            "=",
            _manifest["rows"],
            "|",
            _manifest["sha256"][:16],
        )


# ---------------------------------------------------------------------------
# 8. Denylist dev/local_test et index de similarite deterministe
# ---------------------------------------------------------------------------

UINT64_MASK = (1 << 64) - 1


def hash64(payload: bytes) -> int:
    return int.from_bytes(
        hashlib.blake2b(
            payload,
            digest_size=8,
            person=HASH_PERSON,
        ).digest(),
        "little",
    )


def mix64(value: int) -> int:
    value = (value + 0x9E3779B97F4A7C15) & UINT64_MASK
    value = ((value ^ (value >> 30)) * 0xBF58476D1CE4E5B9) & UINT64_MASK
    value = ((value ^ (value >> 27)) * 0x94D049BB133111EB) & UINT64_MASK
    return (value ^ (value >> 31)) & UINT64_MASK


MINHASH_SEEDS = tuple(
    hash64(f"minhash:{index}".encode("ascii"))
    for index in range(MINHASH_COMPONENTS)
)
MINHASH_SEEDS_ARRAY = np.asarray(MINHASH_SEEDS, dtype=np.uint64)
assert MINHASH_COMPONENTS % MINHASH_BANDS == 0


def text_shingle_hashes(text: str) -> tuple[int, ...]:
    words = text.split()
    if len(words) >= 3:
        grams = (
            "\x1f".join(words[index : index + 3])
            for index in range(len(words) - 2)
        )
    else:
        grams = ("\x1f".join(words),)
    values = sorted(
        {
            hash64(gram.encode("utf-8"))
            for gram in grams
            if gram
        }
    )
    assert values
    return tuple(values)


def minhash_signature(shingles: tuple[int, ...]) -> tuple[int, ...]:
    values = np.asarray(shingles, dtype=np.uint64)[:, None]
    mixed = np.bitwise_xor(values, MINHASH_SEEDS_ARRAY[None, :])
    with np.errstate(over="ignore"):
        mixed = mixed + np.uint64(0x9E3779B97F4A7C15)
        mixed = (
            (mixed ^ (mixed >> np.uint64(30)))
            * np.uint64(0xBF58476D1CE4E5B9)
        )
        mixed = (
            (mixed ^ (mixed >> np.uint64(27)))
            * np.uint64(0x94D049BB133111EB)
        )
        mixed = mixed ^ (mixed >> np.uint64(31))
    return tuple(int(value) for value in mixed.min(axis=0))


def minhash_band_keys(signature: tuple[int, ...]) -> tuple[tuple[int, ...], ...]:
    assert len(signature) == MINHASH_COMPONENTS
    width = MINHASH_COMPONENTS // MINHASH_BANDS
    return tuple(
        (band, *signature[band * width : (band + 1) * width])
        for band in range(MINHASH_BANDS)
    )


def jaccard(left: set[int], right: set[int]) -> float:
    union = left | right
    return len(left & right) / len(union) if union else 1.0


def near_features(
    text: str,
) -> tuple[int, tuple[str, ...], tuple[int, ...], tuple[int, ...]] | None:
    word_count = len(text.split())
    if word_count < MIN_NEAR_WORDS and len(text) < MIN_NEAR_CHARS:
        return None
    shingles = text_shingle_hashes(text)
    return (
        word_count,
        tuple(uregex.findall(r"\p{N}+", text)),
        shingles,
        minhash_signature(shingles),
    )


class NearIndex:
    def __init__(self) -> None:
        self.texts: list[str] = []
        self.word_counts: list[int] = []
        self.number_sequences: list[tuple[str, ...]] = []
        self.buckets: dict[tuple[int, ...], list[int]] = defaultdict(list)
        self.max_bucket_size = 0
        self.pair_checks = 0

    @staticmethod
    def eligible(text: str) -> bool:
        return len(text.split()) >= MIN_NEAR_WORDS or len(text) >= MIN_NEAR_CHARS

    def add(
        self,
        text: str,
        features: tuple[
            int,
            tuple[str, ...],
            tuple[int, ...],
            tuple[int, ...],
        ] | None = None,
    ) -> None:
        features = near_features(text) if features is None else features
        if features is None:
            return
        word_count, numbers, _, signature = features
        text_index = len(self.texts)
        self.texts.append(text)
        self.word_counts.append(word_count)
        self.number_sequences.append(numbers)
        for key in minhash_band_keys(signature):
            bucket = self.buckets[key]
            bucket.append(text_index)
            self.max_bucket_size = max(self.max_bucket_size, len(bucket))
            assert len(bucket) <= 100_000, (
                "LSH_BUCKET_PATHOLOGICAL",
                key[0],
                len(bucket),
            )

    def find(
        self,
        text: str,
        *,
        heldout: bool,
        features: tuple[
            int,
            tuple[str, ...],
            tuple[int, ...],
            tuple[int, ...],
        ] | None = None,
    ) -> int | None:
        features = near_features(text) if features is None else features
        if features is None:
            return None
        word_count, numbers, shingles_tuple, signature = features
        shingles = set(shingles_tuple)
        candidate_ids: set[int] = set()
        for key in minhash_band_keys(signature):
            candidate_ids.update(self.buckets.get(key, ()))

        for candidate_id in sorted(candidate_ids):
            self.pair_checks += 1
            other_word_count = self.word_counts[candidate_id]
            length_ratio = min(word_count, other_word_count) / max(
                1,
                word_count,
                other_word_count,
            )
            if numbers != self.number_sequences[candidate_id]:
                continue
            if heldout:
                minimum_length = 0.90
                minimum_jaccard = HELDOUT_JACCARD
                minimum_edit = HELDOUT_EDIT
            else:
                short = max(word_count, other_word_count) <= 20
                minimum_length = 0.90 if short else 0.85
                minimum_jaccard = (
                    NEAR_SHORT_JACCARD if short else NEAR_LONG_JACCARD
                )
                minimum_edit = NEAR_SHORT_EDIT if short else NEAR_LONG_EDIT
            if length_ratio < minimum_length:
                continue
            other_shingles = set(text_shingle_hashes(self.texts[candidate_id]))
            if jaccard(shingles, other_shingles) < minimum_jaccard:
                continue
            if edit_ratio(text, self.texts[candidate_id]) / 100.0 >= minimum_edit:
                return candidate_id
        return None


def exact_digest(language: str, text: str) -> bytes:
    return hashlib.sha256(f"{language}\0{text}".encode("utf-8")).digest()


def add_heldout_text(
    language: str,
    raw_text: Any,
    exact_values: set[bytes],
    near_index: NearIndex,
    stats: Counter[str],
) -> None:
    normalized, status, _ = normalize_lm_text(raw_text)
    stats[f"normalizer|{status}"] += 1
    if normalized is None:
        stats["rejected"] += 1
        return
    for text in chunk_normalized_text(normalized):
        digest = exact_digest(language, text)
        if digest in exact_values:
            stats["duplicate_exact"] += 1
            continue
        exact_values.add(digest)
        near_index.add(text, near_features(text))
        stats["unique_lines"] += 1


def build_heldout_denylists() -> tuple[
    dict[str, set[bytes]],
    dict[str, NearIndex],
    dict[str, dict[str, Any]],
]:
    exact_by_language = {language: set() for language in LANGUAGES}
    near_by_language = {language: NearIndex() for language in LANGUAGES}
    stats_by_language = {language: Counter() for language in LANGUAGES}

    # Un seul passage sur le manifest V4 pour les six langues.
    parquet_file = pq.ParquetFile(
        V4_MANIFEST["path"],
        memory_map=False,
        pre_buffer=False,
    )
    preferred_text = V4_MANIFEST["text_norm_column"] or V4_MANIFEST["text_column"]
    columns = [
        preferred_text,
        V4_MANIFEST["language_column"],
        V4_MANIFEST["split_column"],
    ]
    rows_seen = 0
    for batch in parquet_file.iter_batches(
        batch_size=8192,
        columns=columns,
        use_threads=False,
    ):
        values = batch.to_pydict()
        rows_seen += batch.num_rows
        for index in range(batch.num_rows):
            if normalize_split(values[columns[2]][index]) == "train":
                continue
            language = canonical_language(values[columns[1]][index])
            assert language in LANGUAGES, (
                "LANGUE_V4_HELDOUT_INCONNUE",
                values[columns[1]][index],
            )
            stats_by_language[language]["v4_parent_rows"] += 1
            add_heldout_text(
                language,
                values[columns[0]][index],
                exact_by_language[language],
                near_by_language[language],
                stats_by_language[language],
            )
    assert rows_seen == V4_MANIFEST["rows_from_footer"]

    # Dev court et eval_long propres a chaque cache V5.
    for language in LANGUAGES:
        cache_root = Path(SELECTED_CACHES[language]["cache_root"])
        for record in SELECTED_CACHES[language]["heldout_text_plan"]:
            path = require_child(record["path"], cache_root)
            parquet_file = pq.ParquetFile(path, memory_map=False, pre_buffer=False)
            columns = record["requested_columns"]
            rows = 0
            for batch in parquet_file.iter_batches(
                batch_size=8192,
                columns=columns,
                use_threads=False,
            ):
                values = batch.to_pydict()
                rows += batch.num_rows
                for raw_text in values[record["text_column"]]:
                    stats_by_language[language][
                        f"{record['heldout_role']}_parent_rows"
                    ] += 1
                    add_heldout_text(
                        language,
                        raw_text,
                        exact_by_language[language],
                        near_by_language[language],
                        stats_by_language[language],
                    )
            assert rows == record["rows_from_footer"]

    summaries = {}
    for language in LANGUAGES:
        digest = hashlib.sha256()
        for value in sorted(exact_by_language[language]):
            digest.update(value)
        summaries[language] = {
            "exact_unique": len(exact_by_language[language]),
            "near_eligible": len(near_by_language[language].texts),
            "exact_set_sha256": digest.hexdigest(),
            "stats": dict(sorted(stats_by_language[language].items())),
            "raw_text_persisted": False,
            "audio_columns_requested": [],
        }
        assert summaries[language]["exact_unique"] > 0
        print(
            "Denylist",
            language,
            "| exact=",
            summaries[language]["exact_unique"],
            "| near=",
            summaries[language]["near_eligible"],
        )
    return exact_by_language, near_by_language, summaries


if RUN_ALREADY_COMPLETE:
    HELDOUT_EXACT: dict[str, set[bytes]] = {}
    HELDOUT_NEAR: dict[str, NearIndex] = {}
    HELDOUT_SUMMARIES: dict[str, dict[str, Any]] = {}
else:
    HELDOUT_EXACT, HELDOUT_NEAR, HELDOUT_SUMMARIES = build_heldout_denylists()


# ---------------------------------------------------------------------------
# 9. Dedup exacte + proche, sortie KenLM deterministe et audit independant
# ---------------------------------------------------------------------------


def update_text_sequence_digest(digest: Any, text: str) -> None:
    encoded = text.encode("utf-8")
    digest.update(len(encoded).to_bytes(8, "little"))
    digest.update(encoded)


def audit_final_language(
    language: str,
    parquet_path: Path,
    gzip_path: Path,
) -> dict[str, Any]:
    parquet_file = pq.ParquetFile(
        parquet_path,
        memory_map=False,
        pre_buffer=False,
    )
    assert parquet_file.schema_arrow.equals(CANDIDATE_SCHEMA, check_metadata=False)
    exact_seen: set[bytes] = set()
    near_seen = NearIndex()
    parquet_sequence = hashlib.sha256()
    rows = 0
    words = 0
    sources: Counter[str] = Counter()
    for batch in parquet_file.iter_batches(
        batch_size=8192,
        columns=CANDIDATE_SCHEMA.names,
        use_threads=False,
    ):
        values = batch.to_pydict()
        for index in range(batch.num_rows):
            text = values["text_norm"][index]
            assert isinstance(text, str) and text
            assert unicodedata.normalize("NFC", text) == text
            assert "\ufffd" not in text
            normalized, _, _ = normalize_lm_text(text)
            assert normalized == text, (language, "normalisation non idempotente")
            assert 1 <= len(text.split()) <= MAX_WORDS_PER_LINE
            assert values["word_count"][index] == len(text.split())
            assert values["char_count"][index] == len(text)
            digest = exact_digest(language, text)
            assert values["exact_sha256"][index] == digest.hex()
            assert digest not in HELDOUT_EXACT[language]
            features = near_features(text)
            assert HELDOUT_NEAR[language].find(
                text,
                heldout=True,
                features=features,
            ) is None
            assert digest not in exact_seen
            assert near_seen.find(
                text,
                heldout=False,
                features=features,
            ) is None
            exact_seen.add(digest)
            near_seen.add(text, features)
            update_text_sequence_digest(parquet_sequence, text)
            sources[values["source_id"][index]] += 1
            rows += 1
            words += len(text.split())
    assert rows == parquet_file.metadata.num_rows and rows > 0

    gzip_sequence = hashlib.sha256()
    gzip_rows = 0
    with gzip.open(gzip_path, "rt", encoding="utf-8", errors="strict", newline="") as handle:
        for line in handle:
            assert line.endswith("\n"), (language, "ligne gzip sans newline")
            text = line[:-1]
            assert text and "\n" not in text and "\r" not in text
            update_text_sequence_digest(gzip_sequence, text)
            gzip_rows += 1
    assert gzip_rows == rows
    assert gzip_sequence.hexdigest() == parquet_sequence.hexdigest()
    return {
        "rows": rows,
        "words": words,
        "sources": dict(sorted(sources.items())),
        "exact_duplicates_detected": 0,
        "near_duplicates_detected": 0,
        "heldout_exact_overlaps": 0,
        "heldout_near_overlaps": 0,
        "replacement_characters": 0,
        "normalization_idempotent": True,
        "parquet_gzip_sequence_sha256": parquet_sequence.hexdigest(),
        "near_index_eligible": len(near_seen.texts),
        "near_pair_checks": near_seen.pair_checks,
        "near_max_bucket_size": near_seen.max_bucket_size,
        "audio_columns_requested": [],
    }


def valid_dedup_checkpoint(language: str) -> dict[str, Any] | None:
    manifest_path = STAGING_RUN_DIR / "manifests" / f"dedup_{language}.json"
    parquet_path = STAGING_RUN_DIR / "final" / f"{language}.parquet"
    gzip_path = STAGING_RUN_DIR / "final" / f"train_{language}.txt.gz"
    try:
        manifest = read_small_json(manifest_path)
        assert manifest["phase"] == "dedup_final"
        assert manifest["language"] == language
        assert manifest["contract_sha256"] == K2_CONTRACT_SHA256
        assert manifest["candidate_sha256"] == CANDIDATE_MANIFESTS[language]["sha256"]
        assert manifest["heldout_exact_set_sha256"] == HELDOUT_SUMMARIES[language][
            "exact_set_sha256"
        ]
        for kind, path in (("parquet", parquet_path), ("gzip", gzip_path)):
            assert path.is_file()
            assert path.stat().st_size == manifest[kind]["bytes"]
            assert sha256_file(path) == manifest[kind]["sha256"]
        assert pq.read_metadata(parquet_path).num_rows == manifest["rows"]
        assert manifest["audit"]["rows"] == manifest["rows"]
        assert manifest["audit"]["heldout_exact_overlaps"] == 0
        assert manifest["audit"]["heldout_near_overlaps"] == 0
        assert manifest["audit"]["exact_duplicates_detected"] == 0
        assert manifest["audit"]["near_duplicates_detected"] == 0
        return manifest
    except Exception:
        return None


def deduplicate_language(language: str) -> dict[str, Any]:
    resumed = valid_dedup_checkpoint(language)
    if resumed is not None:
        print(f"[reprise] dedup {language} : {resumed['rows']} lignes")
        return resumed

    candidate_path = CANDIDATE_PATHS[language]
    assert sha256_file(candidate_path) == CANDIDATE_MANIFESTS[language]["sha256"]
    local_parquet = LOCAL_WORK_DIR / f"final_{language}.parquet"
    local_gzip = LOCAL_WORK_DIR / f"train_{language}.txt.gz"
    for path in (local_parquet, local_gzip):
        if path.exists():
            path.unlink()

    temp_directory = LOCAL_WORK_DIR / f"duckdb_{language}"
    temp_directory.mkdir(parents=True, exist_ok=True)
    connection = duckdb.connect(database=":memory:")
    safe_temp = str(temp_directory).replace("'", "''")
    connection.execute(f"SET temp_directory='{safe_temp}'")
    connection.execute("SET memory_limit='12GB'")
    connection.execute("SET threads=4")
    connection.execute("SET preserve_insertion_order=false")
    order_columns = (
        "source_priority, source_family, source_id, source_revision, "
        "source_path, source_row_id, segment_index, record_id"
    )
    cursor = connection.execute(
        f"SELECT * FROM read_parquet(?) ORDER BY {order_columns}",
        [str(candidate_path)],
    )
    reader = cursor.fetch_record_batch(8192)

    writer = pq.ParquetWriter(
        local_parquet,
        CANDIDATE_SCHEMA,
        compression="zstd",
        use_dictionary=True,
        write_statistics=True,
    )
    accepted_exact: set[bytes] = set()
    accepted_near = NearIndex()
    accepted_buffer: list[dict[str, Any]] = []
    rejection_counts: Counter[str] = Counter()
    accepted_sources: Counter[str] = Counter()
    input_rows = 0
    accepted_rows = 0
    started = time.monotonic()

    def flush_accepted() -> None:
        if not accepted_buffer:
            return
        table = pa.Table.from_pylist(accepted_buffer, schema=CANDIDATE_SCHEMA)
        writer.write_table(table, row_group_size=len(accepted_buffer))
        accepted_buffer.clear()

    with local_gzip.open("wb") as raw_gzip:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            compresslevel=6,
            fileobj=raw_gzip,
            mtime=0,
        ) as gzip_handle:
            for batch in reader:
                assert batch.schema.names == CANDIDATE_SCHEMA.names
                values = batch.to_pydict()
                for index in range(batch.num_rows):
                    input_rows += 1
                    if input_rows % 25_000 == 0:
                        elapsed = (time.monotonic() - started) / 60
                        print(
                            f"  {language} {input_rows}/{CANDIDATE_MANIFESTS[language]['rows']} "
                            f"| gardes {accepted_rows} | {elapsed:.1f} min"
                        )
                    text = values["text_norm"][index]
                    digest = bytes.fromhex(values["exact_sha256"][index])
                    assert digest == exact_digest(language, text)

                    if digest in HELDOUT_EXACT[language]:
                        rejection_counts["heldout_exact"] += 1
                        continue
                    features = near_features(text)
                    if HELDOUT_NEAR[language].find(
                        text,
                        heldout=True,
                        features=features,
                    ) is not None:
                        rejection_counts["heldout_near"] += 1
                        continue
                    if digest in accepted_exact:
                        rejection_counts["duplicate_exact"] += 1
                        continue
                    if accepted_near.find(
                        text,
                        heldout=False,
                        features=features,
                    ) is not None:
                        rejection_counts["duplicate_near"] += 1
                        continue

                    record = {
                        name: values[name][index]
                        for name in CANDIDATE_SCHEMA.names
                    }
                    accepted_buffer.append(record)
                    accepted_exact.add(digest)
                    accepted_near.add(text, features)
                    accepted_sources[record["source_id"]] += 1
                    gzip_handle.write(text.encode("utf-8") + b"\n")
                    accepted_rows += 1
                    if len(accepted_buffer) >= 8192:
                        flush_accepted()

    flush_accepted()
    writer.close()
    connection.close()
    shutil.rmtree(temp_directory, ignore_errors=True)

    assert input_rows == CANDIDATE_MANIFESTS[language]["rows"]
    assert accepted_rows > 0
    assert pq.read_metadata(local_parquet).num_rows == accepted_rows
    assert accepted_rows + sum(rejection_counts.values()) == input_rows
    audit = audit_final_language(language, local_parquet, local_gzip)
    assert audit["rows"] == accepted_rows

    drive_parquet = STAGING_RUN_DIR / "final" / f"{language}.parquet"
    drive_gzip = STAGING_RUN_DIR / "final" / f"train_{language}.txt.gz"
    publish_file(local_parquet, drive_parquet)
    publish_file(local_gzip, drive_gzip)
    manifest = {
        "schema": 1,
        "phase": "dedup_final",
        "language": language,
        "contract_sha256": K2_CONTRACT_SHA256,
        "candidate_sha256": CANDIDATE_MANIFESTS[language]["sha256"],
        "candidate_rows": input_rows,
        "heldout_exact_set_sha256": HELDOUT_SUMMARIES[language][
            "exact_set_sha256"
        ],
        "rows": accepted_rows,
        "rejections": dict(sorted(rejection_counts.items())),
        "accepted_sources": dict(sorted(accepted_sources.items())),
        "lsh": {
            "eligible_rows": len(accepted_near.texts),
            "pair_checks": accepted_near.pair_checks,
            "max_bucket_size": accepted_near.max_bucket_size,
        },
        "parquet": {
            "relative_path": f"final/{language}.parquet",
            "bytes": drive_parquet.stat().st_size,
            "sha256": sha256_file(drive_parquet),
        },
        "gzip": {
            "relative_path": f"final/train_{language}.txt.gz",
            "bytes": drive_gzip.stat().st_size,
            "sha256": sha256_file(drive_gzip),
            "mtime": 0,
        },
        "audit": audit,
        "audio_columns_requested": [],
        "audio_bytes_read": 0,
    }
    atomic_json(
        manifest,
        STAGING_RUN_DIR / "manifests" / f"dedup_{language}.json",
    )
    print(
        "Dedup",
        language,
        "| candidats=",
        input_rows,
        "| final=",
        accepted_rows,
        "| rejets=",
        dict(sorted(rejection_counts.items())),
    )
    del accepted_near, accepted_exact
    gc.collect()
    return manifest


DEDUP_MANIFESTS: dict[str, dict[str, Any]] = {}
if RUN_ALREADY_COMPLETE:
    for _language in LANGUAGES:
        DEDUP_MANIFESTS[_language] = read_small_json(
            FINAL_RUN_DIR / "manifests" / f"dedup_{_language}.json"
        )
else:
    for _language in LANGUAGES:
        DEDUP_MANIFESTS[_language] = deduplicate_language(_language)


# ---------------------------------------------------------------------------
# 10. Publication atomique du corpus textuel V6 et rapport sans contenu brut
# ---------------------------------------------------------------------------


def atomic_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".tmp-{os.getpid()}")
    frame.to_csv(temporary, index=False, encoding="utf-8", lineterminator="\n")
    os.replace(temporary, path)
    print("csv  ->", path)


def artifact_metadata(path: Path) -> dict[str, Any]:
    assert path.is_file() and path.stat().st_size > 0
    return {
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }


if RUN_ALREADY_COMPLETE:
    COMPLETE = read_small_json(FINAL_RUN_DIR / "_COMPLETE.json")
    K2_REPORT_FINAL = read_small_json(
        FINAL_RUN_DIR / "kenlm_text_corpus_20_K2.json"
    )
    assert K2_REPORT_FINAL["status"] == "PASS_READY_FOR_20_K3_KENLM_BUILD"
    print("[reprise] 20.K2 etait deja complet et a ete reverifie.")
else:
    language_rows = []
    for language in LANGUAGES:
        candidate = CANDIDATE_MANIFESTS[language]
        final = DEDUP_MANIFESTS[language]
        audit = final["audit"]
        assert candidate["audio_bytes_read"] == 0
        assert final["audio_bytes_read"] == 0
        assert candidate["audio_columns_requested"] == []
        assert final["audio_columns_requested"] == []
        assert audit["exact_duplicates_detected"] == 0
        assert audit["near_duplicates_detected"] == 0
        assert audit["heldout_exact_overlaps"] == 0
        assert audit["heldout_near_overlaps"] == 0
        assert audit["replacement_characters"] == 0
        assert audit["normalization_idempotent"] is True
        language_rows.append(
            {
                "language": language,
                "candidate_rows": candidate["rows"],
                "final_rows": final["rows"],
                "final_words": audit["words"],
                "removed_rows": candidate["rows"] - final["rows"],
                "heldout_exact_removed": final["rejections"].get(
                    "heldout_exact", 0
                ),
                "heldout_near_removed": final["rejections"].get(
                    "heldout_near", 0
                ),
                "duplicate_exact_removed": final["rejections"].get(
                    "duplicate_exact", 0
                ),
                "duplicate_near_removed": final["rejections"].get(
                    "duplicate_near", 0
                ),
                "heldout_denylist_lines": HELDOUT_SUMMARIES[language][
                    "exact_unique"
                ],
                "parquet_mib": round(final["parquet"]["bytes"] / 2**20, 3),
                "kenlm_text_gzip_mib": round(final["gzip"]["bytes"] / 2**20, 3),
            }
        )
    language_summary = pd.DataFrame(language_rows)
    assert set(language_summary.language) == set(LANGUAGES)
    assert (language_summary.final_rows > 0).all()

    remote_audits = {}
    for key, remote in sorted(REMOTE_PLAN.items()):
        language = remote["language"]
        audit = CANDIDATE_MANIFESTS[language]["source_audits"].get(key)
        assert audit is not None, ("REMOTE_SOURCE_NOT_EXTRACTED", key)
        assert audit["audio_columns_requested"] == []
        remote_audits[key] = audit

    source_attribution = {
        "local_cache_text": {
            "type": "audited_local_cache",
            "languages": list(LANGUAGES),
            "selection_gate": "20.K1 PASS",
            "local_text_mode": {
                language: SELECTED_CACHES[language]["local_text_mode"]
                for language in LANGUAGES
            },
            "cache_complete_sha256": {
                language: SELECTED_CACHES[language]["cache_complete_sha256"]
                for language in LANGUAGES
            },
            "upstream_provenance": "See each immutable cache _COMPLETE.json",
        },
        "local_v4_train": {
            "type": "audited_local_manifest",
            "languages": list(LANGUAGES),
            "manifest_sha256": V4_MANIFEST["sha256"],
        },
        "remote": {
            key: {
                "repo_id": record["repo_id"],
                "revision": record["revision"],
                "path": record["path"],
                "reported_licenses": record["reported_licenses"],
                "license_status": record["license_status"],
                "remote_identity": record["remote_identity"],
            }
            for key, record in sorted(REMOTE_PLAN.items())
        },
    }

    K2_REPORT_FINAL = {
        "schema": 1,
        "cell": "20.K2",
        "status": "PASS_READY_FOR_20_K3_KENLM_BUILD",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "run_id": K2_RUN_ID,
        "contract_sha256": K2_CONTRACT_SHA256,
        "k1_run_id": K1_LATEST["run_id"],
        "k1_report_sha256": EXPECTED_K1_REPORT_SHA256,
        "text_only": True,
        "runtime_environment": {
            "python_version": sys.version.split()[0],
            "package_versions": PACKAGE_VERSIONS,
        },
        "audio_columns_requested": [],
        "audio_files_opened": [],
        "audio_bytes_read": 0,
        "normalizer": NORMALIZER_CONTRACT,
        "normalizer_calibration": NORMALIZER_CALIBRATION,
        "heldout_denylists": HELDOUT_SUMMARIES,
        "candidate_manifests": CANDIDATE_MANIFESTS,
        "intermediate_candidate_files_removed": True,
        "dedup_manifests": DEDUP_MANIFESTS,
        "remote_extraction_audits": remote_audits,
        "source_attribution": source_attribution,
        "language_summary": language_rows,
        "totals": {
            "candidate_rows": int(language_summary.candidate_rows.sum()),
            "final_rows": int(language_summary.final_rows.sum()),
            "final_words": int(language_summary.final_words.sum()),
            "removed_rows": int(language_summary.removed_rows.sum()),
        },
        "reports_contain_raw_corpus_text": False,
        "next_step": "20.K3 — construire et evaluer les KenLM V6 par langue/domaine",
    }
    report_path = STAGING_RUN_DIR / "kenlm_text_corpus_20_K2.json"
    summary_path = STAGING_RUN_DIR / "language_summary_20_K2.csv"
    atomic_json(K2_REPORT_FINAL, report_path)
    atomic_csv(language_summary, summary_path)

    # Les candidats contiennent une copie intermediaire du texte. Ils sont
    # supprimes seulement apres validation de toutes les sorties finales.
    shutil.rmtree(STAGING_RUN_DIR / "candidates", ignore_errors=True)

    artifact_relatives = [
        "contract.json",
        "kenlm_text_corpus_20_K2.json",
        "language_summary_20_K2.csv",
        *[f"final/{language}.parquet" for language in LANGUAGES],
        *[f"final/train_{language}.txt.gz" for language in LANGUAGES],
        *[f"manifests/candidate_{language}.json" for language in LANGUAGES],
        *[f"manifests/dedup_{language}.json" for language in LANGUAGES],
    ]
    artifacts = {
        relative: artifact_metadata(STAGING_RUN_DIR / relative)
        for relative in artifact_relatives
    }
    COMPLETE = {
        "schema": 1,
        "cell": "20.K2",
        "status": "PASS_READY_FOR_20_K3_KENLM_BUILD",
        "finished_at": datetime.now(timezone.utc).isoformat(),
        "run_id": K2_RUN_ID,
        "contract_sha256": K2_CONTRACT_SHA256,
        "k1_report_sha256": EXPECTED_K1_REPORT_SHA256,
        "text_only": True,
        "audio_bytes_read": 0,
        "languages": list(LANGUAGES),
        "artifacts": artifacts,
    }
    atomic_json(COMPLETE, STAGING_RUN_DIR / "_COMPLETE.json")

    assert not FINAL_RUN_DIR.exists()
    os.replace(STAGING_RUN_DIR, FINAL_RUN_DIR)
    verified_complete = validate_completed_run(FINAL_RUN_DIR)
    assert verified_complete is not None


# Repare aussi les pointeurs de rapport si la session precedente s'est coupee
# juste apres la publication atomique du corpus.
assert validate_completed_run(FINAL_RUN_DIR) is not None
report_run_dir = REPORT_ROOT / K2_RUN_ID
report_run_dir.mkdir(parents=True, exist_ok=True)
for filename in (
    "kenlm_text_corpus_20_K2.json",
    "language_summary_20_K2.csv",
    "_COMPLETE.json",
):
    publish_file(FINAL_RUN_DIR / filename, report_run_dir / filename)
latest = {
    "schema": 1,
    "cell": "20.K2",
    "status": "PASS_READY_FOR_20_K3_KENLM_BUILD",
    "run_id": K2_RUN_ID,
    "contract_sha256": K2_CONTRACT_SHA256,
    "report": str(report_run_dir / "kenlm_text_corpus_20_K2.json"),
    "report_sha256": sha256_file(
        report_run_dir / "kenlm_text_corpus_20_K2.json"
    ),
    "complete": str(FINAL_RUN_DIR / "_COMPLETE.json"),
    "complete_sha256": sha256_file(FINAL_RUN_DIR / "_COMPLETE.json"),
    "complete_mirror": str(report_run_dir / "_COMPLETE.json"),
    "corpus_root": str(FINAL_RUN_DIR),
    "audio_bytes_read": 0,
}
atomic_json(latest, REPORT_ROOT / "LATEST_ATTEMPT.json")
atomic_json(latest, REPORT_ROOT / "LATEST.json")
atomic_json(latest, REPORT_ROOT / "LATEST_PASS.json")


print("\n=== CORPUS KENLM V6 20.K2 ===")
print(pd.DataFrame(K2_REPORT_FINAL["language_summary"]).to_string(index=False))
print("STATUT 20.K2 :", K2_REPORT_FINAL["status"])
print("Corpus       :", FINAL_RUN_DIR)
print("Run ID       :", K2_RUN_ID)
print("SHA contrat  :", K2_CONTRACT_SHA256)
print("Audio lu     : 0 octet")
print("Etape suivante : 20.K3 — construction/evaluation KenLM V6")
